# NUEVAS CELDAS — Agregar al notebook existente después de la celda 9 (limpieza)
## Estas celdas asumen que `df_clean` ya existe con todas las transformaciones previas.
---

## A. Filtrado por los 4 tipos de cáncer seleccionados

In [ ]:
# ── Códigos CIE-10 de los 4 tipos de cáncer del estudio ──────────────────────
CANCER_MAP = {
    "C50":    "Mama",
    "C18":    "Colorrectal",
    "C19":    "Colorrectal",
    "C20":    "Colorrectal",
    "C53":    "Cuello uterino",
    "C34":    "Pulmón",
}

df_clean["diag_3"] = df_clean["DIAGNOSTICO1"].astype(str).str[:3].str.upper()
df_clean["tipo_cancer"] = df_clean["diag_3"].map(CANCER_MAP)

df_onco = df_clean[df_clean["tipo_cancer"].notna()].copy()

# ── Variable de mortalidad intrahospitalaria ──────────────────────────────────
df_onco["muerte_intra"] = df_onco["TIPOALTA"].astype(str).str.upper().str.contains(
    "FALLEC", na=False
).astype(int)

# ── Identificador de paciente unificado (2024 usa nombre distinto) ────────────
id_col = "CIP_ENCRIPTADO" if "CIP_ENCRIPTADO" in df_onco.columns else "ID_BENEFICIARIO"

print(f"Identificador de paciente usado: '{id_col}'")
print(f"Total egresos oncológicos seleccionados: {len(df_onco):,}")
print(f"Pacientes únicos en estos egresos:       {df_onco[id_col].nunique():,}")
print()
print("Distribución por tipo de cáncer (egresos):")
print(df_onco["tipo_cancer"].value_counts().to_string())
print()
print("Distribución por tipo de cáncer (pacientes únicos):")
print(df_onco.groupby("tipo_cancer")[id_col].nunique().sort_values(ascending=False).to_string())

## B1. Pacientes únicos por tipo de cáncer, sexo y grupo etario

In [ ]:
# Usamos nunique sobre el identificador, agrupado por tipo_cancer + grupo_edad + SEXO
# para representar personas, no ingresos.

df_onco["grupo_edad"] = pd.cut(
    df_onco["edad"],
    bins=[0, 20, 30, 40, 50, 60, 70, 80, 120],
    labels=["<20", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"],
    right=False
)

tipos = ["Mama", "Colorrectal", "Cuello uterino", "Pulmón"]
fig, axes = plt.subplots(1, 4, figsize=(20, 7), sharey=True)

for ax, tipo in zip(axes, tipos):
    sub = df_onco[df_onco["tipo_cancer"] == tipo].copy()
    sub = sub[sub["SEXO"].isin(["HOMBRE", "MUJER"])]

    # Conteo de pacientes únicos por grupo_edad y SEXO
    pyr = (
        sub.groupby(["grupo_edad", "SEXO"], observed=True)[id_col]
        .nunique()
        .unstack(fill_value=0)
    )
    y_pos = range(len(pyr))

    if "HOMBRE" in pyr.columns:
        ax.barh(list(y_pos), [-v for v in pyr["HOMBRE"]], color=COLORES[0],
                label="Hombre", height=0.8, edgecolor="white")
    if "MUJER" in pyr.columns:
        ax.barh(list(y_pos), list(pyr["MUJER"]), color=COLORES[3],
                label="Mujer", height=0.8, edgecolor="white")

    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(pyr.index.astype(str), fontsize=9)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Cáncer de\n{tipo}", fontweight="bold")
    ax.set_xlabel("Pacientes únicos")
    if ax == axes[0]:
        ax.legend(fontsize=8, loc="lower left")
    # Formatear eje x como positivo en ambos lados
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(int(x)):,}"))

fig.suptitle("Pirámide de pacientes únicos por tipo de cáncer, sexo y grupo etario (2019–2024)",
             fontsize=13, fontweight="bold", y=1.02)
sns.despine()
plt.tight_layout()
plt.show()

## B2. Diversidad de procedimientos por tipo de cáncer

In [ ]:
# Distribución de n_procedimientos por tipo de cáncer — sobre egresos individuales.
# n_procedimientos ya fue calculado en la celda de limpieza original.

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, tipo in zip(axes, tipos):
    sub = df_onco[df_onco["tipo_cancer"] == tipo]["n_procedimientos"].dropna()
    mediana = sub.median()
    media   = sub.mean()

    ax.hist(sub, bins=range(0, int(sub.max()) + 2), color=COLORES[0],
            edgecolor="white", alpha=0.85)
    ax.axvline(mediana, color="red",    linestyle="--", linewidth=2,
               label=f"Mediana: {mediana:.0f}")
    ax.axvline(media,   color="orange", linestyle="--", linewidth=2,
               label=f"Media: {media:.1f}")
    ax.set_title(f"Cáncer de {tipo}", fontweight="bold")
    ax.set_xlabel("Número de procedimientos por egreso")
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.suptitle("Distribución de procedimientos por egreso según tipo de cáncer",
             fontsize=13, fontweight="bold")
sns.despine()
plt.tight_layout()
plt.show()

print("\nEstadísticas de procedimientos por tipo de cáncer:")
display(
    df_onco.groupby("tipo_cancer")["n_procedimientos"]
    .describe().round(2)
)

## B3. Heatmap de procedimientos CIE-9 más frecuentes por tipo de cáncer

In [ ]:
# Top procedimientos CIE-9 por tipo de cáncer (sobre egresos)
proc_cols = [c for c in df_onco.columns if c.startswith("PROCEDIMIENTO")]

TOP_N = 10
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for ax, tipo in zip(axes, tipos):
    sub = df_onco[df_onco["tipo_cancer"] == tipo]
    # Apilar todos los procedimientos en una sola serie
    all_procs = (
        sub[proc_cols]
        .stack()
        .reset_index(drop=True)
        .astype(str)
        .str.strip()
    )
    all_procs = all_procs[all_procs.notna() & (all_procs != "nan") & (all_procs != "")]
    top = all_procs.value_counts().head(TOP_N)

    ax.barh(top.index[::-1], top.values[::-1], color=COLORES[2], edgecolor="white")
    ax.set_title(f"Cáncer de {tipo}\nTop {TOP_N} procedimientos CIE-9",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Frecuencia")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.tick_params(axis="y", labelsize=8)

sns.despine()
plt.suptitle("Procedimientos CIE-9 más frecuentes por tipo de cáncer",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## C1. Intensidad de tratamiento por hospital: clasificación en terciles

In [ ]:
# Para cada tipo de cáncer, calculamos el promedio de procedimientos
# por hospital y lo clasificamos en terciles: baja / media / alta intensidad.
# Umbral mínimo: 30 pacientes únicos por hospital-tipo.

MIN_PACIENTES = 30

intensidad_frames = []
for tipo in tipos:
    sub = df_onco[df_onco["tipo_cancer"] == tipo].copy()
    hosp_stats = (
        sub.groupby("COD_HOSPITAL")
        .agg(
            n_pacientes=(id_col, "nunique"),
            media_proc=("n_procedimientos", "mean"),
            mediana_proc=("n_procedimientos", "median"),
            mortalidad=("muerte_intra", "mean"),
            mediana_estada=("dias_estada", "median"),
        )
        .reset_index()
    )
    hosp_stats = hosp_stats[hosp_stats["n_pacientes"] >= MIN_PACIENTES].copy()

    # Terciles de intensidad de procedimientos
    hosp_stats["intensidad"] = pd.qcut(
        hosp_stats["media_proc"],
        q=3,
        labels=["Baja", "Media", "Alta"]
    )
    hosp_stats["tipo_cancer"] = tipo
    intensidad_frames.append(hosp_stats)

df_intensidad = pd.concat(intensidad_frames, ignore_index=True)

print("Hospitales clasificados por intensidad de procedimientos:")
print(df_intensidad.groupby(["tipo_cancer", "intensidad"]).size().to_string())
print()
print("Promedio de procedimientos por grupo de intensidad:")
display(
    df_intensidad.groupby(["tipo_cancer", "intensidad"])["media_proc"]
    .mean().round(2).unstack()
)

## C2. Mortalidad por grupo de intensidad de tratamiento

In [ ]:
# Gráfico central del estudio: ¿mayor intensidad → más o menos mortalidad?
# Tasa de mortalidad media por grupo de intensidad, para cada tipo de cáncer.

fig, ax = plt.subplots(figsize=(14, 6))

mort_intensidad = (
    df_intensidad.groupby(["tipo_cancer", "intensidad"])["mortalidad"]
    .mean()
    .reset_index()
)
mort_intensidad["mortalidad_pct"] = mort_intensidad["mortalidad"] * 100

x = np.arange(len(tipos))
width = 0.25
colores_int = {"Baja": COLORES[2], "Media": COLORES[4], "Alta": COLORES[1]}

for i, (nivel, color) in enumerate(colores_int.items()):
    vals = [
        mort_intensidad[
            (mort_intensidad["tipo_cancer"] == t) &
            (mort_intensidad["intensidad"] == nivel)
        ]["mortalidad_pct"].values
        for t in tipos
    ]
    vals = [v[0] if len(v) > 0 else 0 for v in vals]
    bars = ax.bar(x + i * width, vals, width, label=f"Intensidad {nivel}",
                  color=color, edgecolor="white")
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.1,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(tipos, fontsize=11)
ax.set_ylabel("Tasa de mortalidad intrahospitalaria media (%)")
ax.set_title(
    "Mortalidad intrahospitalaria según intensidad de procedimientos, por tipo de cáncer",
    fontweight="bold"
)
ax.legend(title="Grupo de hospital", fontsize=10)
sns.despine()
plt.tight_layout()
plt.show()

print("\nTabla resumen: mortalidad media por tipo de cáncer e intensidad:")
display(mort_intensidad.pivot(index="tipo_cancer", columns="intensidad",
                              values="mortalidad_pct").round(2))

## C3. Días de estada por grupo de intensidad

In [ ]:
# Asociamos el grupo de intensidad del hospital al dataframe de egresos
# para poder graficar la distribución de días de estada por grupo.

hosp_intensidad_map = df_intensidad.set_index(["tipo_cancer", "COD_HOSPITAL"])["intensidad"].to_dict()

df_onco["intensidad_hosp"] = [
    hosp_intensidad_map.get((tc, h), np.nan)
    for tc, h in zip(df_onco["tipo_cancer"], df_onco["COD_HOSPITAL"])
]

df_plot_int = df_onco[df_onco["intensidad_hosp"].notna()].copy()

fig, axes = plt.subplots(1, 4, figsize=(20, 6), sharey=False)
for ax, tipo in zip(axes, tipos):
    sub = df_plot_int[df_plot_int["tipo_cancer"] == tipo]
    sns.boxplot(
        data=sub,
        x="intensidad_hosp",
        y="dias_estada",
        order=["Baja", "Media", "Alta"],
        palette=[COLORES[2], COLORES[4], COLORES[1]],
        showfliers=False,
        ax=ax,
        width=0.55
    )
    ax.set_title(f"Cáncer de {tipo}", fontweight="bold")
    ax.set_xlabel("Intensidad del hospital")
    ax.set_ylabel("Días de estada" if tipo == tipos[0] else "")

fig.suptitle(
    "Días de estada según intensidad de procedimientos del hospital, por tipo de cáncer",
    fontsize=13, fontweight="bold"
)
sns.despine()
plt.tight_layout()
plt.show()

## D1. Prueba de Kruskal-Wallis: ¿varía n_procedimientos entre hospitales?

In [ ]:
from scipy.stats import kruskal

alpha = 0.05
print("=" * 65)
print("KRUSKAL-WALLIS: Variabilidad de procedimientos entre hospitales")
print("=" * 65)
print(f"{'Tipo de cáncer':<20} {'H-stat':>10} {'p-value':>12} {'Resultado':>20}")
print("-" * 65)

for tipo in tipos:
    sub = df_onco[df_onco["tipo_cancer"] == tipo].copy()
    grupos = [
        g["n_procedimientos"].dropna().values
        for _, g in sub.groupby("COD_HOSPITAL")
        if g["n_procedimientos"].dropna().shape[0] >= 30
    ]
    if len(grupos) < 2:
        print(f"{tipo:<20} {'N/A':>10} {'N/A':>12} {'Insuficientes grupos':>20}")
        continue
    h, p = kruskal(*grupos)
    resultado = "Rechaza H0 (p<0.05)" if p < alpha else "No rechaza H0"
    print(f"{tipo:<20} {h:>10.2f} {p:>12.2e} {resultado:>20}")

print("\nInterpretación: Si se rechaza H0, la distribución de procedimientos"
      "\ndifiere significativamente entre hospitales para ese tipo de cáncer.")

## D2. Regresión logística: mortalidad ~ procedimientos + severidad + hospital

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

print("=" * 65)
print("REGRESIÓN LOGÍSTICA: muerte_intra ~ procedimientos + severidad + edad")
print("=" * 65)

for tipo in tipos:
    sub = df_onco[
        (df_onco["tipo_cancer"] == tipo) &
        df_onco[["muerte_intra", "n_procedimientos", "severidad", "edad"]].notna().all(axis=1)
    ].copy()

    if len(sub) < 100:
        print(f"\n{tipo}: muestra insuficiente ({len(sub)} registros).")
        continue

    # Excluir hospitales con muy pocos casos para efectos fijos
    min_hosp = sub.groupby("COD_HOSPITAL").size()
    hosps_validos = min_hosp[min_hosp >= 30].index
    sub = sub[sub["COD_HOSPITAL"].isin(hosps_validos)].copy()

    # Convertir hospital a dummies (sin categoría de referencia explícita → drop_first)
    sub["COD_HOSPITAL"] = sub["COD_HOSPITAL"].astype(str)

    try:
        formula = "muerte_intra ~ n_procedimientos + severidad + edad + C(COD_HOSPITAL)"
        modelo = smf.logit(formula, data=sub).fit(disp=False, maxiter=200)

        # Mostrar solo variables de interés (no todos los hospitales)
        coef_tabla = modelo.summary2().tables[1]
        vars_interes = ["n_procedimientos", "severidad", "edad"]
        print(f"\n{'─'*65}")
        print(f"Cáncer de {tipo}  (n = {len(sub):,}, hospitales = {sub['COD_HOSPITAL'].nunique()})")
        print(f"{'─'*65}")
        display(
            coef_tabla.loc[
                coef_tabla.index.isin(vars_interes),
                ["Coef.", "Std.Err.", "z", "P>|z|", "[0.025", "0.975]"]
            ].round(4)
        )
        print(f"  AIC: {modelo.aic:.1f}  |  Pseudo-R²: {modelo.prsquared:.4f}")

        # Odds ratios para n_procedimientos
        import numpy as np
        or_proc = np.exp(modelo.params["n_procedimientos"])
        ic_low  = np.exp(modelo.conf_int().loc["n_procedimientos", 0])
        ic_high = np.exp(modelo.conf_int().loc["n_procedimientos", 1])
        print(f"  OR n_procedimientos: {or_proc:.3f}  IC95%: [{ic_low:.3f}, {ic_high:.3f}]")

    except Exception as e:
        print(f"  Error en {tipo}: {e}")

## D3. Regresión OLS: días de estada ~ procedimientos + severidad + hospital

In [ ]:
import statsmodels.formula.api as smf

print("=" * 65)
print("REGRESIÓN OLS: dias_estada ~ procedimientos + severidad + edad")
print("=" * 65)

ols_results = {}
for tipo in tipos:
    sub = df_onco[
        (df_onco["tipo_cancer"] == tipo) &
        df_onco[["dias_estada", "n_procedimientos", "severidad", "edad"]].notna().all(axis=1)
    ].copy()

    if len(sub) < 100:
        print(f"\n{tipo}: muestra insuficiente.")
        continue

    min_hosp = sub.groupby("COD_HOSPITAL").size()
    hosps_validos = min_hosp[min_hosp >= 30].index
    sub = sub[sub["COD_HOSPITAL"].isin(hosps_validos)].copy()
    sub["COD_HOSPITAL"] = sub["COD_HOSPITAL"].astype(str)

    try:
        formula = "dias_estada ~ n_procedimientos + severidad + edad + C(COD_HOSPITAL)"
        modelo = smf.ols(formula, data=sub).fit()
        ols_results[tipo] = modelo

        coef_tabla = modelo.summary2().tables[1]
        vars_interes = ["n_procedimientos", "severidad", "edad"]
        print(f"\n{'─'*65}")
        print(f"Cáncer de {tipo}  (n = {len(sub):,}, hospitales = {sub['COD_HOSPITAL'].nunique()})")
        print(f"{'─'*65}")
        display(
            coef_tabla.loc[
                coef_tabla.index.isin(vars_interes),
                ["Coef.", "Std.Err.", "t", "P>|t|", "[0.025", "0.975]"]
            ].round(4)
        )
        print(f"  R² ajustado: {modelo.rsquared_adj:.4f}  |  AIC: {modelo.aic:.1f}")

    except Exception as e:
        print(f"  Error en {tipo}: {e}")

## D4. Visualización de efectos fijos por hospital (OLS)

In [ ]:
# Para un tipo de cáncer a la vez, graficamos los coeficientes de hospital
# del modelo OLS: qué hospitales tienen estadas sistemáticamente más largas/cortas
# que la referencia, controlando por severidad y procedimientos.

tipo_grafico = "Colorrectal"  # Cambiar al tipo que más interese

if tipo_grafico in ols_results:
    modelo = ols_results[tipo_grafico]
    params = modelo.params
    conf   = modelo.conf_int()

    hosp_params = params[params.index.str.startswith("C(COD_HOSPITAL)")].copy()
    hosp_conf   = conf[conf.index.str.startswith("C(COD_HOSPITAL)")].copy()

    hosp_params.index = hosp_params.index.str.replace(r"C\(COD_HOSPITAL\)T\.", "",
                                                       regex=True)
    hosp_conf.index   = hosp_params.index

    # Ordenar por coeficiente
    orden = hosp_params.sort_values().index
    vals  = hosp_params[orden]
    low   = hosp_conf.loc[orden, 0]
    high  = hosp_conf.loc[orden, 1]

    # Mostrar solo los 20 más extremos (10 más y 10 menos días)
    n_mostrar = min(20, len(orden))
    idx = list(orden[:n_mostrar // 2]) + list(orden[-n_mostrar // 2:])
    vals_plot = vals[idx]
    low_plot  = low[idx]
    high_plot = high[idx]

    fig, ax = plt.subplots(figsize=(10, max(6, len(idx) * 0.4)))
    y_pos = range(len(idx))
    ax.barh(list(y_pos), vals_plot.values,
            xerr=[vals_plot.values - low_plot.values, high_plot.values - vals_plot.values],
            color=[COLORES[1] if v > 0 else COLORES[2] for v in vals_plot.values],
            edgecolor="white", capsize=4, alpha=0.85)
    ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(idx, fontsize=8)
    ax.set_xlabel("Días de estada adicionales respecto al hospital de referencia")
    ax.set_title(
        f"Efectos fijos por hospital — Cáncer de {tipo_grafico}\n"
        f"(Top/Bottom {n_mostrar // 2} hospitales, controlando por severidad y procedimientos)",
        fontweight="bold"
    )
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print(f"Modelo OLS no disponible para {tipo_grafico}.")

## D5. Evolución temporal de mortalidad por tipo de cáncer (pacientes únicos)
*(Incorpora el feedback de usar identificadores únicos en lugar de ingresos)*

In [ ]:
# Tasa de mortalidad anual por tipo de cáncer, calculada sobre pacientes únicos.
# Se cuenta un paciente como fallecido si tuvo AL MENOS UN egreso con TIPOALTA=FALLECIDO.

mort_anual_frames = []
for tipo in tipos:
    sub = df_onco[df_onco["tipo_cancer"] == tipo].copy()
    # Por paciente y año: ¿tuvo algún egreso con fallecimiento?
    por_paciente_anio = (
        sub.groupby([id_col, "anio"])
        .agg(fallecio=("muerte_intra", "max"))
        .reset_index()
    )
    tasa = (
        por_paciente_anio.groupby("anio")
        .agg(
            n_pacientes=(id_col, "count"),
            n_fallecidos=("fallecio", "sum")
        )
        .assign(tasa_mort=lambda d: d["n_fallecidos"] / d["n_pacientes"] * 100)
        .reset_index()
    )
    tasa["tipo_cancer"] = tipo
    mort_anual_frames.append(tasa)

df_mort_anual = pd.concat(mort_anual_frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(13, 6))
for tipo, color in zip(tipos, [COLORES[0], COLORES[1], COLORES[3], COLORES[2]]):
    data = df_mort_anual[df_mort_anual["tipo_cancer"] == tipo]
    ax.plot(data["anio"], data["tasa_mort"], "o-",
            color=color, linewidth=2.5, markersize=8, label=tipo)

ax.axvspan(2020, 2021, alpha=0.08, color="red", label="Período COVID-19")
ax.set_xlabel("Año")
ax.set_ylabel("Tasa de mortalidad intrahospitalaria (% pacientes únicos)")
ax.set_title(
    "Evolución de la mortalidad intrahospitalaria por tipo de cáncer (2019–2024)\n"
    "Calculada sobre pacientes únicos por año",
    fontweight="bold"
)
ax.legend(title="Tipo de cáncer", fontsize=10)
ax.set_xticks(sorted(df_mort_anual["anio"].unique()))
sns.despine()
plt.tight_layout()
plt.show()